In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from adjustText import adjust_text
import matplotlib.patheffects as pe
import os
import glob
os.chdir(r'C:\Users\andre\Documents\GitHub\gut-liver-TRM\Geneformer')

In [3]:
csv_files = glob.glob("*.csv")

for getting a good mapping table, I use feature.tsv that generated along with the scRNAseq data (for this study we used that mapping logic derived from 10x 2020-A human reference)
(Human GRCh38 (GENCODE v32/Ensembl98))

If this feature.tsv is not available to the data (like, I got processed data but not the output from cellranger)

I use ensembl https://www.ensembl.org/biomart/martview/05575bee58978a75d31c351f728fae29 and a reference table from github workshop

In [56]:
mart_txid = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\mart_export.txt")
del mart_txid['Transcript stable ID']
mart_txid = mart_txid.drop_duplicates(subset = 'Gene stable ID')
# mart_txid.index = mart_txid['Gene stable ID']
mart_txid['Gene name and ID'] = mart_txid['Gene name'].astype(str).map(str) +'_'+ mart_txid['Gene stable ID'].astype(str).map(str)
mart_txid['Gene name and ID'][mart_txid['Gene name'].isna()] = mart_txid['Gene stable ID'].astype(str)[mart_txid['Gene name'].isna()]

# txid = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\tx2gene_grch38_ens94.txt",sep = '\t',index_col = 1)

txid = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\features_10x_2020A.tsv",sep = '\t',names = ['Gene stable ID','Gene name','Gene Expression'])
del txid['Gene Expression']
txid['Used by 10X'] = True

ultimate_txid = pd.merge(left = mart_txid, right = txid, left_on = 'Gene stable ID', right_on = 'Gene stable ID',how = 'outer',suffixes =['_mart','_10x'])
ultimate_txid['ultimate_gene_name'] = ultimate_txid['Gene name_10x']
ultimate_txid['ultimate_gene_name'][ultimate_txid['ultimate_gene_name'].isna()] = ultimate_txid['Gene name_mart'][ultimate_txid['ultimate_gene_name'].isna()]
ultimate_txid['ultimate_gene_name'][ultimate_txid['ultimate_gene_name'].isna()] = ultimate_txid['Gene name and ID'][ultimate_txid['ultimate_gene_name'].isna()]
ultimate_txid_clean = ultimate_txid[['Gene stable ID','Used by 10X','ultimate_gene_name']]
ultimate_txid_clean.to_csv('ultimate_txid_clean_10x2020A.csv')
ultimate_txid_clean.index = ultimate_txid_clean['Gene stable ID']
id_table = ultimate_txid_clean['ultimate_gene_name']
id_dict = id_table.to_dict()#index = False)

In [65]:
print(ultimate_txid.groupby(['Gene name_mart'])['Used by 10X'].count().index[ultimate_txid.groupby(['Gene name_mart'])['Used by 10X'].count()])

Index(['5S_rRNA', '5S_rRNA', '5S_rRNA', '5_8S_rRNA', '5_8S_rRNA', '5_8S_rRNA',
       '5_8S_rRNA', '5_8S_rRNA', '5_8S_rRNA', '5_8S_rRNA',
       ...
       '5S_rRNA', '5_8S_rRNA', '5_8S_rRNA', '5S_rRNA', '5_8S_rRNA',
       '5_8S_rRNA', '5_8S_rRNA', '5_8S_rRNA', '5S_rRNA', '5S_rRNA'],
      dtype='object', name='Gene name_mart', length=41164)


In [ ]:
for csv_file in csv_files:
    df = pd.read_csv(csv_file,index_col= 3)
    df = df.drop(columns = 'Unnamed: 0')

    data = df.rename(index = id_dict)
    data = data[data['Shift_to_goal_end'] > 0 ]

    # Calculate -log10(FDR)
    data['-log10_FDR'] = -np.log10(data['Goal_end_FDR'])